# Roster and Stat Scraper for Elite Prospects Site
- because CHN's roster data is not always accurate I want to get a second source to check it against.
- after some very simple exploration on hockeydb.com I was flagged and IP banned for scraping, though there site TOS says they allow limited scraping of their data for non commercial use.
    - the commented out code is some that was developed for hockeydb.com but not tested bvery much
- after being IP banned (it was removed a few hours later) I decided to look for another source and found elit prospects which seems more open to the type of scraping I am trying

In [1]:
# Updated cleaner with multi-position handling: Pos_OVR, Pos_Prime, Pos_Second
import pandas as pd
import numpy as np
import re

elite_path = "../TEMP//elite_prospects_current_team_roster_temp.csv"
elite = pd.read_csv(elite_path)

ALLOWED_POS = {"G","D","F","LW","C","RW"}

def parse_player(player_str: str):
    """Extract (name, position_text, captaincy) from the Elite Prospects Player field."""
    if not isinstance(player_str, str):
        return None, None, None
    cap = re.search(r'\"([AC])\"', player_str)
    captaincy = cap.group(1) if cap else None
    cleaned = re.sub(r'\s*\"[AC]\"\s*', '', player_str)
    posm = re.search(r'\(([^)]+)\)\s*$', cleaned)
    position_text = posm.group(1).strip().upper() if posm else None
    name = re.sub(r'\s*\([^)]+\)\s*$', '', cleaned).strip()
    return name, position_text, captaincy

def split_name(full_name: str):
    if not isinstance(full_name, str) or not full_name.strip():
        return None, None
    parts = full_name.strip().split()
    if len(parts) == 1:
        return parts[0], None
    return " ".join(parts[:-1]), parts[-1]

def ht_to_inches(ht_str: str):
    if not isinstance(ht_str, str):
        return None
    m = re.match(r"^\s*(\d+)'\s*(\d+)\"\s*$", ht_str)
    if not m:
        return None
    return int(m.group(1))*12 + int(m.group(2))

def split_birthplace(bp: str):
    if not isinstance(bp, str):
        return None, None, None
    parts = [p.strip() for p in str(bp).split(",")]
    if len(parts) == 3:
        city, sp, country = parts
    elif len(parts) == 2:
        city, country = parts; sp = None
    else:
        city, sp, country = None, None, parts[0] if parts else None
    return city, sp, country

def normalize_positions(position_text: str):
    """
    Returns Pos_OVR, Pos_Prime, Pos_Second:
      - Pos_OVR: {G,D,F} (C/LW/RW/F -> F)
      - Pos_Prime: first of [G,D,F,LW,C,RW]
      - Pos_Second: second token if present; else None
    """
    if not isinstance(position_text, str) or not position_text.strip():
        return None, None, None
    tokens = [t.strip().upper() for t in position_text.split("/") if t.strip()]
    tokens = [t for t in tokens if t in ALLOWED_POS]
    if not tokens:
        return None, None, None

    pos_prime = tokens[0]
    pos_second = tokens[1] if len(tokens) > 1 else None

    if pos_prime == "G":
        pos_ovr = "G"
    elif pos_prime == "D":
        pos_ovr = "D"
    else:
        pos_ovr = "F"
    return pos_ovr, pos_prime, pos_second

def clean_elite_prospects(df: pd.DataFrame) -> pd.DataFrame:
    # 1) drop junk + section headers
    df = df.drop(columns=[c for c in ['Unnamed: 0','N'] if c in df.columns], errors='ignore').copy()
    headers = {'GOALTENDERS','DEFENSEMEN','FORWARDS'}
    df = df[~df['Player'].isin(headers)].copy()

    # 2) parse Player → (name, positions, captaincy)
    df[['Full_Name','Position_text','Captaincy']] = pd.DataFrame(
        df['Player'].apply(parse_player).tolist(), index=df.index
    )

    # 3) split names
    df[['First_Name','Last_Name']] = pd.DataFrame(
        df['Full_Name'].apply(split_name).tolist(), index=df.index
    )

    # 4) jersey No
    df['No'] = (
        df.get('#', pd.Series(index=df.index, dtype='object')).astype(str)
          .str.replace('#','', regex=False).str.strip()
          .replace({'nan': None, 'None': None, '': None})
    )

    # 5) height/weight/dob
    df['Height_Inches'] = df['HT'].apply(ht_to_inches)
    df['DOB'] = df['Born'].astype(str).replace({'nan': None})
    df['Wt'] = pd.to_numeric(df['WT'], errors='coerce')
    df['Ht'] = df['HT']
    df['Shoots'] = df['S'] if 'S' in df.columns else None

    # 6) birthplace
    df[['City','State_Province','Country']] = pd.DataFrame(
        df['Birthplace'].apply(split_birthplace).tolist(), index=df.index
    )
    df['Hometown'] = df['Birthplace']

    # 7) multi-position mapping
    df[['Pos_OVR','Pos_Prime','Pos_Second']] = pd.DataFrame(
        df['Position_text'].apply(normalize_positions).tolist(), index=df.index
    )
    # keep Position for compatibility with your existing tools
    df['Position'] = df['Pos_Prime']

    # 8) placeholders to match your schema
    for c in ['Yr','Draft_Year','NHL_Team','D_Round','Last Team','League','Current Team']:
        if c not in df.columns:
            df[c] = None

    # 9) final column order
    target_cols = [
        'Current Team','Last_Name','First_Name','No',
        'Position','Pos_OVR','Pos_Prime','Pos_Second',
        'Yr','Ht','Wt','DOB','Hometown','Height_Inches',
        'Draft_Year','NHL_Team','D_Round','Last Team','League',
        'City','State_Province','Country','Captaincy','Shoots'
    ]
    for c in target_cols:
        if c not in df.columns:
            df[c] = None
    return df[target_cols].reset_index(drop=True)

# Example usage on your uploaded sample:
cleaned = clean_elite_prospects(elite)


out_path = "../TEMP/DATA/elite_prospects_cleaned_with_positions.csv"
cleaned.to_csv(out_path, index=False)
out_path



'../TEMP/DATA/elite_prospects_cleaned_with_positions.csv'

In [2]:
## Display Cleaned Sample
cleaned.head(10)

,Current Team,Last_Name,First_Name,No,Position,Pos_OVR,Pos_Prime,Pos_Second,Yr,Ht,...,Draft_Year,NHL_Team,D_Round,Last Team,League,City,State_Province,Country,Captaincy,Shoots
0,None,Augustine,Trey,1,G,G,G,None,None,"6'1""",...,None,None,None,None,None,South Lyon,MI,USA,None,L
1,None,Gilbert,Dolan,30,G,G,G,None,None,"6'2""",...,None,None,None,None,None,South Bend,IN,USA,None,L
2,None,Strahl,Melvin,32,G,G,G,None,None,"6'3""",...,None,None,None,None,None,Sollefteå,None,SWE,None,L
3,None,Barnhill,Sean,3,D,D,D,None,None,"6'6""",...,None,None,None,None,None,Scottsdale,AZ,USA,None,R
4,None,Basgall,Matt,9,D,D,D,None,None,"5'10""",...,None,None,None,None,None,Lake Forest,IL,USA,C,R
5,None,Geary,Patrick,2,D,D,D,None,None,"6'1""",...,None,None,None,None,None,Hamburg,NY,USA,A,L
6,None,Lahey,Matthew,14,D,D,D,None,None,"6'6""",...,None,None,None,None,None,Victoria,BC,CAN,None,L
7,None,Ralph,Colin,4,D,D,D,None,None,"6'5""",...,None,None,None,None,None,Maple Grove,MN,USA,None,L
8,None,Shoudy,Travis,5,D,D,D,None,None,"5'10""",...,None,None,None,None,None,Marysville,MI,USA,A,L
9,None,Strbak,Maxim,8,D,D,D,None,None,"6'2""",...,None,None,None,None,None,Kosice,None,SVK,None,R


In [3]:
# Create Table of teams and roster URLs

# Load school info including links to EP.com
school_info_df = pd.read_csv('../data/school_info/arena_school_info.csv')

# Load and check
school_info_df.head()

,Team,Arena,Capacity,Sheet_length,Sheet_width,School,Latitude,Longitude,hex1,hex2,hex3,simp_color,logo_abv,abv,ncaa_name,ncaa_data_alts,eliteprospects_url
0,Air Force,Cadet Ice Arena,2470,200,85,Air Force,39.013739,-104.883727,3087,8a8d8f,NaN,NaN,afa,Air Force,Air Force,"AIRFOR, Air Force",https://www.eliteprospects.com/team/2453/air-f...
1,Alaska,Carlson Center,4595,200,100,Alaska,64.842124,-147.763841,236192,ffcd00,NaN,NaN,akf,Alaska,Alas Fairbanks,"AK FBK, Alas. Fairbanks",https://www.eliteprospects.com/team/2071/univ....
2,Alaska Anchorage,Avis Alaska Sports Complex,800,200,85,Alaska-Anchorage,61.205536,-149.872737,00583d,ffc425,NaN,NaN,aka,UAA,Alas Anchorage,"AK ANC, Alas. Anchorage",https://www.eliteprospects.com/team/1915/univ....
3,American Intl,MassMutual Center,6866,200,85,American Int'l,42.118003,-72.554326,0,ffb60f,NaN,NaN,aic,AIC,American Intl,"AM INT, American Int'l",https://www.eliteprospects.com/team/1252/ameri...
4,American Int'l,MassMutual Center,6866,200,85,American Int'l,42.118003,-72.554326,0,ffb60f,NaN,NaN,aic,AIC,American Intl,"AM INT, American Int'l",https://www.eliteprospects.com/team/1252/ameri...


In [4]:
# simplify to just team name and roster URL
school_info_df = school_info_df[['Team', 'eliteprospects_url']]
# Save to temp csv file
school_info_df.to_csv('../TEMP/school_info_temp.csv', index=False)

In [5]:
# # JUPYTER TEST BLOCK — EliteProspects D1 Roster Scraper (polite, 3 random teams)
# # Requirements: pandas, requests, beautifulsoup4, lxml
# import os, time, random, logging, re
# from pathlib import Path
# import pandas as pd
# import requests
# from bs4 import BeautifulSoup

# # -----------------------------
# # Config & folders
# # -----------------------------
# BASE = Path("../TEMP/")
# RAW_HTML_DIR = BASE / "raw_html"
# RAW_CSV_DIR  = BASE / "raw_csv"
# CLEAN_DIR    = BASE / "clean_csv"
# LOG_DIR      = BASE / "logs"
# OUT_DIR      = BASE / "out"
# for d in [RAW_HTML_DIR, RAW_CSV_DIR, CLEAN_DIR, LOG_DIR, OUT_DIR]:
#     d.mkdir(parents=True, exist_ok=True)

# logging.basicConfig(
#     filename=LOG_DIR / "ep_roster_scrape.log",
#     level=logging.INFO,
#     format="%(asctime)s %(levelname)s %(message)s"
# )

# HEADERS = {
#     "User-Agent": "NCAADataSauce Roster Builder (research use; contact: bakedmitten@gmail.com)",
#     "From": "youremail@example.com",
#     "Accept-Language": "en-US,en;q=0.9",
# }

# # -----------------------------
# # Cleaner (with Pos_OVR / Pos_Prime / Pos_Second)
# # -----------------------------
# ALLOWED_POS = {"G","D","F","LW","C","RW"}

# def parse_player(player_str: str):
#     if not isinstance(player_str, str):
#         return None, None, None
#     cap = re.search(r'\"([AC])\"', player_str)
#     captaincy = cap.group(1) if cap else None
#     cleaned = re.sub(r'\s*\"[AC]\"\s*', '', player_str)
#     posm = re.search(r'\(([^)]+)\)\s*$', cleaned)
#     position_text = posm.group(1).strip().upper() if posm else None
#     name = re.sub(r'\s*\([^)]+\)\s*$', '', cleaned).strip()
#     return name, position_text, captaincy

# def split_name(full_name: str):
#     if not isinstance(full_name, str) or not full_name.strip():
#         return None, None
#     parts = full_name.strip().split()
#     if len(parts) == 1:
#         return parts[0], None
#     return " ".join(parts[:-1]), parts[-1]

# def ht_to_inches(ht_str: str):
#     if not isinstance(ht_str, str):
#         return None
#     m = re.match(r"^\s*(\d+)'\s*(\d+)\"\s*$", ht_str)
#     if not m:
#         return None
#     return int(m.group(1))*12 + int(m.group(2))

# def split_birthplace(bp: str):
#     if not isinstance(bp, str):
#         return None, None, None
#     parts = [p.strip() for p in str(bp).split(",")]
#     if len(parts) == 3:
#         city, sp, country = parts
#     elif len(parts) == 2:
#         city, country = parts; sp = None
#     else:
#         city, sp, country = None, None, parts[0] if parts else None
#     return city, sp, country

# def normalize_positions(position_text: str):
#     if not isinstance(position_text, str) or not position_text.strip():
#         return None, None, None
#     tokens = [t.strip().upper() for t in position_text.split("/") if t.strip()]
#     tokens = [t for t in tokens if t in ALLOWED_POS]
#     if not tokens:
#         return None, None, None
#     pos_prime = tokens[0]
#     pos_second = tokens[1] if len(tokens) > 1 else None
#     if pos_prime == "G":
#         pos_ovr = "G"
#     elif pos_prime == "D":
#         pos_ovr = "D"
#     else:
#         pos_ovr = "F"
#     return pos_ovr, pos_prime, pos_second

# def clean_elite_prospects(df: pd.DataFrame) -> pd.DataFrame:
#     df = df.drop(columns=[c for c in ['Unnamed: 0','N'] if c in df.columns], errors='ignore').copy()
#     headers = {'GOALTENDERS','DEFENSEMEN','FORWARDS'}
#     if 'Player' in df.columns:
#         df = df[~df['Player'].isin(headers)].copy()

#     df[['Full_Name','Position_text','Captaincy']] = pd.DataFrame(
#         df['Player'].apply(parse_player).tolist(), index=df.index
#     )
#     df[['First_Name','Last_Name']] = pd.DataFrame(
#         df['Full_Name'].apply(split_name).tolist(), index=df.index
#     )
#     df['No'] = (
#         df.get('#', pd.Series(index=df.index, dtype='object')).astype(str)
#           .str.replace('#','', regex=False).str.strip()
#           .replace({'nan': None, 'None': None, '': None})
#     )
#     df['Height_Inches'] = df['HT'].apply(ht_to_inches) if 'HT' in df.columns else None
#     df['DOB'] = df['Born'].astype(str).replace({'nan': None}) if 'Born' in df.columns else None
#     df['Wt'] = pd.to_numeric(df['WT'], errors='coerce') if 'WT' in df.columns else None
#     df['Ht'] = df['HT'] if 'HT' in df.columns else None
#     df['Shoots'] = df['S'] if 'S' in df.columns else None

#     if 'Birthplace' in df.columns:
#         df[['City','State_Province','Country']] = pd.DataFrame(
#             df['Birthplace'].apply(split_birthplace).tolist(), index=df.index
#         )
#         df['Hometown'] = df['Birthplace']
#     else:
#         for c in ['City','State_Province','Country','Hometown']:
#             df[c] = None

#     df[['Pos_OVR','Pos_Prime','Pos_Second']] = pd.DataFrame(
#         df['Position_text'].apply(normalize_positions).tolist(), index=df.index
#     )
#     df['Position'] = df['Pos_Prime']

#     for c in ['Yr','Draft_Year','NHL_Team','D_Round','Last Team','League','Current Team']:
#         if c not in df.columns:
#             df[c] = None

#     target_cols = [
#         'Current Team','Last_Name','First_Name','No',
#         'Position','Pos_OVR','Pos_Prime','Pos_Second',
#         'Yr','Ht','Wt','DOB','Hometown','Height_Inches',
#         'Draft_Year','NHL_Team','D_Round','Last Team','League',
#         'City','State_Province','Country','Captaincy','Shoots'
#     ]
#     for c in target_cols:
#         if c not in df.columns:
#             df[c] = None
#     return df[target_cols].reset_index(drop=True)

# # -----------------------------
# # Fetch + Parse
# # -----------------------------
# def polite_sleep(min_s: float, max_s: float):
#     time.sleep(random.uniform(min_s, max_s))

# def fetch_html(url: str, session: requests.Session, refresh: bool, cache_path: Path):
#     if cache_path.exists() and not refresh:
#         return cache_path.read_text(encoding="utf-8", errors="ignore")
#     r = session.get(url, headers=HEADERS, timeout=30)
#     if r.status_code == 429:
#         logging.warning(f"429 Too Many Requests for {url}.")
#         raise RuntimeError("429")
#     r.raise_for_status()
#     html = r.text
#     cache_path.write_text(html, encoding="utf-8")
#     return html

# def parse_roster_table(html: str) -> pd.DataFrame:
#     try:
#         tables = pd.read_html(html, flavor="bs4")
#     except ValueError:
#         tables = []

#     candidate = None
#     for t in tables:
#         cols = [c if isinstance(c, str) else c[0] for c in t.columns]
#         if "Player" in cols and ("Born" in cols or "HT" in cols):
#             candidate = t
#             break

#     if candidate is not None:
#         candidate.columns = [str(c).strip() for c in candidate.columns]
#         return candidate

#     from bs4 import BeautifulSoup
#     soup = BeautifulSoup(html, "html.parser")
#     table = soup.find("table")
#     if table is None:
#         raise ValueError("No table found in page.")
#     df = pd.read_html(str(table))[0]
#     df.columns = [str(c).strip() for c in df.columns]
#     return df

# # -----------------------------
# # Team CSV helpers (your format: Team, eliteprospects_url)
# # -----------------------------
# def read_teams_csv(csv_path: str) -> pd.DataFrame:
#     teams = pd.read_csv(csv_path)
#     if not {'Team','eliteprospects_url'}.issubset(set(teams.columns)):
#         raise ValueError("CSV must contain columns: 'Team' and 'eliteprospects_url'")
#     teams = teams.rename(columns={'eliteprospects_url': 'EP_Roster_URL'})
#     return teams[['Team','EP_Roster_URL']]

# def pick_random_subset(teams_df: pd.DataFrame, n: int = 3, seed: int | None = None) -> pd.DataFrame:
#     return teams_df.sample(n=n, random_state=seed).reset_index(drop=True)

# # -----------------------------
# # Main builder (from a given teams DF)
# # -----------------------------
# def build_master_from_df(teams_df: pd.DataFrame, sleep_min: float = 8.0, sleep_max: float = 15.0, refresh: bool = False):
#     all_clean = []
#     session = requests.Session()

#     for _, row in teams_df.iterrows():
#         team = row["Team"]
#         url = row["EP_Roster_URL"]
#         safe_team = re.sub(r'[^A-Za-z0-9]+','_', team).strip("_")
#         html_path = RAW_HTML_DIR / f"{safe_team}.html"
#         raw_csv_path = RAW_CSV_DIR / f"{safe_team}.csv"
#         clean_path   = CLEAN_DIR / f"{safe_team}_clean.csv"

#         try:
#             html = fetch_html(url, session, refresh, html_path)
#             df_raw = parse_roster_table(html)
#             df_raw.to_csv(raw_csv_path, index=False)

#             df_clean = clean_elite_prospects(df_raw)
#             df_clean["Current Team"] = team
#             df_clean.to_csv(clean_path, index=False)

#             all_clean.append(df_clean)
#             logging.info(f"OK: {team} ({len(df_clean)} rows)")
#         except RuntimeError as e:
#             if "429" in str(e):
#                 logging.warning("Hit 429. Backing off for 15 minutes.")
#                 time.sleep(15 * 60)
#                 continue
#             else:
#                 logging.exception(f"Runtime error for {team}")
#         except Exception as e:
#             logging.exception(f"Failed for {team}: {e}")

#         polite_sleep(sleep_min, sleep_max)

#     if all_clean:
#         master = pd.concat(all_clean, ignore_index=True)
#         out_partial = OUT_DIR / "all_d1_master.partial.csv"
#         master.to_csv(out_partial, index=False)
#         return master
#     else:
#         return pd.DataFrame()

# # -----------------------------
# # TEST RUN: pick 3 random teams from your CSV and build
# # -----------------------------
# TEAM_CSV = "../TEMP/school_info_temp.csv"  # update path if needed
# # TEMP\school_info_temp.csv
# teams_df = read_teams_csv(TEAM_CSV)
# subset_df = pick_random_subset(teams_df, n=3, seed=None)  # set seed for reproducibility if desired
# print("Selected teams:")
# print(subset_df)

# master_df = build_master_from_df(subset_df, sleep_min=8.0, sleep_max=15.0, refresh=False)
# print(f"Collected {len(master_df)} rows from {len(subset_df)} teams.")
# display(master_df.head(20))


In [6]:
# Create an improved test block addressing: 
# - FutureWarning (wrap HTML in StringIO)
# - More robust table detection & column normalization
# - Better error surfacing (prints + logs)
# - Auto-ensure '?tab=roster' on EP URLs when missing
#
# The cell reads your 'school_info_temp.csv', samples 3 teams, and runs.

# JUPYTER TEST BLOCK v2 — EliteProspects Roster Scraper (robust parse, 3 random teams)
# Requirements: pandas, requests, beautifulsoup4, lxml
import os, time, random, logging, re
from pathlib import Path
from io import StringIO
import pandas as pd
import requests
from bs4 import BeautifulSoup


# -----------------------------
# Config & folders
# -----------------------------
BASE = Path("../TEMP/")
RAW_HTML_DIR = BASE / "raw_html"
RAW_CSV_DIR  = BASE / "raw_csv"
CLEAN_DIR    = BASE / "clean_csv"
LOG_DIR      = BASE / "logs"
OUT_DIR      = BASE / "out"
for d in [RAW_HTML_DIR, RAW_CSV_DIR, CLEAN_DIR, LOG_DIR, OUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)
# # -----------------------------
# # Config & folders
# # -----------------------------
# BASE = Path(".")
# RAW_HTML_DIR = BASE / "raw_html"
# RAW_CSV_DIR  = BASE / "raw_csv"
# CLEAN_DIR    = BASE / "clean_csv"
# LOG_DIR      = BASE / "logs"
# OUT_DIR      = BASE / "out"
# for d in [RAW_HTML_DIR, RAW_CSV_DIR, CLEAN_DIR, LOG_DIR, OUT_DIR]:
#     d.mkdir(parents=True, exist_ok=True)

logger = logging.getLogger("ep_rosters")
logger.setLevel(logging.INFO)
# Log to file + console
fh = logging.FileHandler(LOG_DIR / "ep_roster_scrape.log")
fh.setLevel(logging.INFO)
ch = logging.StreamHandler()
ch.setLevel(logging.INFO)
fmt = logging.Formatter("%(asctime)s %(levelname)s %(message)s")
fh.setFormatter(fmt); ch.setFormatter(fmt)
# Avoid duplicate handlers in repeated runs
if not logger.handlers:
    logger.addHandler(fh); logger.addHandler(ch)

HEADERS = {
    "User-Agent": "NCAADataSauce Roster Builder (research use; contact: youremail@example.com)",
    "From": "youremail@example.com",
    "Accept-Language": "en-US,en;q=0.9",
}

# -----------------------------
# Cleaner (with Pos_OVR / Pos_Prime / Pos_Second)
# -----------------------------
ALLOWED_POS = {"G","D","F","LW","C","RW"}

def parse_player(player_str: str):
    if not isinstance(player_str, str):
        return None, None, None
    cap = re.search(r'\"([AC])\"', player_str)
    captaincy = cap.group(1) if cap else None
    cleaned = re.sub(r'\s*\"[AC]\"\s*', '', player_str)
    posm = re.search(r'\(([^)]+)\)\s*$', cleaned)
    position_text = posm.group(1).strip().upper() if posm else None
    name = re.sub(r'\s*\([^)]+\)\s*$', '', cleaned).strip()
    return name, position_text, captaincy

def split_name(full_name: str):
    if not isinstance(full_name, str) or not full_name.strip():
        return None, None
    parts = full_name.strip().split()
    if len(parts) == 1:
        return parts[0], None
    return " ".join(parts[:-1]), parts[-1]

def ht_to_inches(ht_str: str):
    if not isinstance(ht_str, str):
        return None
    m = re.match(r"^\s*(\d+)'\s*(\d+)\"\s*$", ht_str)
    if not m:
        return None
    return int(m.group(1))*12 + int(m.group(2))

def split_birthplace(bp: str):
    if not isinstance(bp, str):
        return None, None, None
    parts = [p.strip() for p in str(bp).split(",")]
    if len(parts) == 3:
        city, sp, country = parts
    elif len(parts) == 2:
        city, country = parts; sp = None
    else:
        city, sp, country = None, None, parts[0] if parts else None
    return city, sp, country

def normalize_positions(position_text: str):
    if not isinstance(position_text, str) or not position_text.strip():
        return None, None, None
    tokens = [t.strip().upper() for t in position_text.split("/") if t.strip()]
    tokens = [t for t in tokens if t in ALLOWED_POS]
    if not tokens:
        return None, None, None
    pos_prime = tokens[0]
    pos_second = tokens[1] if len(tokens) > 1 else None
    pos_ovr = "G" if pos_prime == "G" else ("D" if pos_prime == "D" else "F")
    return pos_ovr, pos_prime, pos_second

def clean_elite_prospects(df: pd.DataFrame) -> pd.DataFrame:
    df = df.drop(columns=[c for c in ['Unnamed: 0','N'] if c in df.columns], errors='ignore').copy()
    headers = {'GOALTENDERS','DEFENSEMEN','FORWARDS'}
    if 'Player' in df.columns:
        df = df[~df['Player'].isin(headers)].copy()

    df[['Full_Name','Position_text','Captaincy']] = pd.DataFrame(
        df['Player'].apply(parse_player).tolist(), index=df.index
    )
    df[['First_Name','Last_Name']] = pd.DataFrame(
        df['Full_Name'].apply(split_name).tolist(), index=df.index
    )
    df['No'] = (
        df.get('#', pd.Series(index=df.index, dtype='object')).astype(str)
          .str.replace('#','', regex=False).str.strip()
          .replace({'nan': None, 'None': None, '': None})
    )
    df['Height_Inches'] = df['HT'].apply(ht_to_inches) if 'HT' in df.columns else None
    df['DOB'] = df['Born'].astype(str).replace({'nan': None}) if 'Born' in df.columns else None
    df['Wt'] = pd.to_numeric(df['WT'], errors='coerce') if 'WT' in df.columns else None
    df['Ht'] = df['HT'] if 'HT' in df.columns else None
    df['Shoots'] = df['S'] if 'S' in df.columns else None

    if 'Birthplace' in df.columns:
        df[['City','State_Province','Country']] = pd.DataFrame(
            df['Birthplace'].apply(split_birthplace).tolist(), index=df.index
        )
        df['Hometown'] = df['Birthplace']
    else:
        for c in ['City','State_Province','Country','Hometown']:
            df[c] = None

    df[['Pos_OVR','Pos_Prime','Pos_Second']] = pd.DataFrame(
        df['Position_text'].apply(normalize_positions).tolist(), index=df.index
    )
    df['Position'] = df['Pos_Prime']

    for c in ['Yr','Draft_Year','NHL_Team','D_Round','Last Team','League','Current Team']:
        if c not in df.columns:
            df[c] = None

    target_cols = [
        'Current Team','Last_Name','First_Name','No',
        'Position','Pos_OVR','Pos_Prime','Pos_Second',
        'Yr','Ht','Wt','DOB','Hometown','Height_Inches',
        'Draft_Year','NHL_Team','D_Round','Last Team','League',
        'City','State_Province','Country','Captaincy','Shoots'
    ]
    for c in target_cols:
        if c not in df.columns:
            df[c] = None
    return df[target_cols].reset_index(drop=True)

# -----------------------------
# Robust parsing helpers
# -----------------------------
def flatten_cols(cols):
    out = []
    for c in cols:
        if isinstance(c, tuple):
            c = " ".join([str(x) for x in c if str(x).lower() != "nan"]).strip()
        out.append(str(c).strip())
    return out

def normalize_header_name(name: str) -> str:
    n = name.strip().lower()
    # Map common header variants to our expected ones
    if n in {"player","players","name"}: return "Player"
    if n in {"born","birth","birthdate","date of birth"}: return "Born"
    if n in {"ht","height"}: return "HT"
    if n in {"wt","weight"}: return "WT"
    if n in {"s","shoots","shot"}: return "S"
    if n in {"ctrct","contract"}: return "CTRCT"
    if n in {"#","no","no."}: return "#"
    if n in {"birth place","birthplace","place of birth"}: return "Birthplace"
    return name.strip()

def coerce_column_names(df: pd.DataFrame) -> pd.DataFrame:
    # Flatten any MultiIndex and normalize known synonyms
    df = df.copy()
    df.columns = flatten_cols(df.columns)
    df.columns = [normalize_header_name(c) for c in df.columns]
    return df

def looks_like_roster(df: pd.DataFrame) -> bool:
    cols = set([c.lower() for c in df.columns])
    return ("player" in cols) and (("born" in cols) or ("ht" in cols) or ("birthplace" in cols))

def parse_roster_table(html: str) -> pd.DataFrame:
    # Quick guard: detect bot/cookie walls
    low = html.lower()
    if any(s in low for s in ["enable javascript", "captcha", "cloudflare", "access denied"]):
        raise ValueError("Page appears to be a bot/cookie wall (no roster table in HTML).")

    # Preferred: pandas.read_html on a StringIO (avoids FutureWarning)
    tables = []
    try:
        tables = pd.read_html(StringIO(html))
    except ValueError:
        pass

    candidates = []
    for t in tables:
        t = coerce_column_names(t)
        if looks_like_roster(t) and len(t) >= 5:
            candidates.append(t)

    if candidates:
        # Choose the widest (most columns), tie-break by length
        candidates.sort(key=lambda d: (d.shape[1], d.shape[0]), reverse=True)
        return candidates[0]

    # Fallback: use BeautifulSoup to find the specific table that contains a "Player" header
    soup = BeautifulSoup(html, "html.parser")
    for tbl in soup.find_all("table"):
        df = pd.read_html(StringIO(str(tbl)))[0]
        df = coerce_column_names(df)
        if looks_like_roster(df) and len(df) >= 5:
            return df

    raise ValueError("No roster-like table found.")

# -----------------------------
# Fetch + Parse
# -----------------------------
def polite_sleep(min_s: float, max_s: float):
    time.sleep(random.uniform(min_s, max_s))

def fix_url(url: str) -> str:
    """Ensure we're loading the roster tab; if already has query params, append safely."""
    try:
        from urllib.parse import urlparse, parse_qs, urlencode, urlunparse
        u = urlparse(url)
        q = parse_qs(u.query)
        if 'tab' not in q:
            q['tab'] = ['roster']
        new_q = urlencode({k: v[0] if isinstance(v, list) else v for k, v in q.items()})
        return urlunparse((u.scheme, u.netloc, u.path, u.params, new_q, u.fragment))
    except Exception:
        return url

def fetch_html(url: str, session: requests.Session, refresh: bool, cache_path: Path):
    if cache_path.exists() and not refresh:
        return cache_path.read_text(encoding="utf-8", errors="ignore")
    r = session.get(url, headers=HEADERS, timeout=30)
    if r.status_code == 429:
        logger.warning(f"429 Too Many Requests for {url}.")
        raise RuntimeError("429")
    r.raise_for_status()
    html = r.text
    cache_path.write_text(html, encoding="utf-8")
    return html

# -----------------------------
# Team CSV helpers (your format: Team, eliteprospects_url)
# -----------------------------
def read_teams_csv(csv_path: str) -> pd.DataFrame:
    teams = pd.read_csv(csv_path)
    need = {'Team','eliteprospects_url'}
    if not need.issubset(set(teams.columns)):
        raise ValueError("CSV must contain columns: 'Team' and 'eliteprospects_url'")
    teams = teams.rename(columns={'eliteprospects_url': 'EP_Roster_URL'})
    teams['EP_Roster_URL'] = teams['EP_Roster_URL'].astype(str).apply(fix_url)
    return teams[['Team','EP_Roster_URL']]

def pick_random_subset(teams_df: pd.DataFrame, n: int = 3, seed: int | None = None) -> pd.DataFrame:
    return teams_df.sample(n=n, random_state=seed).reset_index(drop=True)

# -----------------------------
# Main builder (from a given teams DF)
# -----------------------------
def build_master_from_df(teams_df: pd.DataFrame, sleep_min: float = 8.0, sleep_max: float = 15.0, refresh: bool = False):
    all_clean = []
    session = requests.Session()

    for _, row in teams_df.iterrows():
        team = row["Team"]
        url = row["EP_Roster_URL"]
        safe_team = re.sub(r'[^A-Za-z0-9]+','_', team).strip("_")
        html_path = RAW_HTML_DIR / f"{safe_team}.html"
        raw_csv_path = RAW_CSV_DIR / f"{safe_team}.csv"
        clean_path   = CLEAN_DIR / f"{safe_team}_clean.csv"

        try:
            html = fetch_html(url, session, refresh, html_path)
            df_raw = parse_roster_table(html)
            df_raw.to_csv(raw_csv_path, index=False)

            if 'Player' not in df_raw.columns:
                raise ValueError(f"Parsed table missing 'Player' column. Columns={list(df_raw.columns)}")

            df_clean = clean_elite_prospects(df_raw)
            df_clean["Current Team"] = team
            df_clean.to_csv(clean_path, index=False)

            all_clean.append(df_clean)
            logger.info(f"OK: {team} ({len(df_clean)} rows)")
        except Exception as e:
            logger.exception(f"FAILED: {team} -> {e}")
        polite_sleep(sleep_min, sleep_max)

    if all_clean:
        master = pd.concat(all_clean, ignore_index=True)
        out_partial = OUT_DIR / "all_d1_master.partial.csv"
        master.to_csv(out_partial, index=False)
        return master
    else:
        return pd.DataFrame()

# # -----------------------------
# # TEST RUN: pick 3 random teams from your CSV and build
# # -----------------------------
# TEAM_CSV = "../TEMP/school_info_temp.csv"  # update path if needed
# teams_df = read_teams_csv(TEAM_CSV)
# subset_df = pick_random_subset(teams_df, n=3, seed=None)  # set seed for reproducibility if desired
# print("Selected teams:")
# print(subset_df)

# master_df = build_master_from_df(subset_df, sleep_min=8.0, sleep_max=15.0, refresh=False)
# print(f"Collected {len(master_df)} rows from {len(subset_df)} teams.")
# display(master_df.head(20))

# '''
# path = "/mnt/data/ep_roster_scrape_block_v2.py"
# with open(path, "w", encoding="utf-8") as f:
#     f.write(improved_code)

# path


In [7]:
# Run for all teams on NCAA D1 List and save to CSV
TEAM_CSV = "../TEMP/school_info_temp.csv"  # update path if needed
teams_df = read_teams_csv(TEAM_CSV)
subset_df = pick_random_subset(teams_df, n=3, seed=None)  # set seed for reproducibility if desired
print("Selected teams:")
# print(subset_df)
print(teams_df)

master_df = build_master_from_df(teams_df, sleep_min=5.0, sleep_max=10.0, refresh=False)
print(f"Collected {len(master_df)} rows from {len(subset_df)} teams.")
display(master_df.head(20))

Selected teams:
                Team                                      EP_Roster_URL
0          Air Force  https://www.eliteprospects.com/team/2453/air-f...
1             Alaska  https://www.eliteprospects.com/team/2071/univ....
2   Alaska Anchorage  https://www.eliteprospects.com/team/1915/univ....
3      American Intl  https://www.eliteprospects.com/team/1252/ameri...
4     American Int'l  https://www.eliteprospects.com/team/1252/ameri...
..               ...                                                ...
64             Union  https://www.eliteprospects.com/team/1366/union...
65           Vermont  https://www.eliteprospects.com/team/710/univ.-...
66  Western Michigan  https://www.eliteprospects.com/team/1250/weste...
67         Wisconsin  https://www.eliteprospects.com/team/452/univ.-...
68              Yale  https://www.eliteprospects.com/team/786/yale-u...

[69 rows x 2 columns]


2025-08-27 16:34:10,668 INFO OK: Air Force (32 rows)
2025-08-27 16:34:19,237 INFO OK: Alaska (30 rows)
2025-08-27 16:34:27,023 INFO OK: Alaska Anchorage (29 rows)
2025-08-27 16:34:34,966 INFO OK: American Intl (33 rows)
2025-08-27 16:34:42,450 INFO OK: American Int'l (33 rows)
2025-08-27 16:34:48,533 INFO OK: Arizona State (27 rows)
2025-08-27 16:34:56,849 INFO OK: Army (30 rows)
2025-08-27 16:35:04,266 INFO OK: Augustana (29 rows)
2025-08-27 16:35:14,324 INFO OK: Bemidji State (32 rows)
2025-08-27 16:35:20,259 INFO OK: Bentley (31 rows)
2025-08-27 16:35:27,602 INFO OK: Boston College (28 rows)
2025-08-27 16:35:34,248 INFO OK: Boston University (19 rows)
2025-08-27 16:35:42,485 INFO OK: Bowling Green (28 rows)
2025-08-27 16:35:49,832 INFO OK: Brown (28 rows)
2025-08-27 16:35:59,841 INFO OK: Canisius (30 rows)
2025-08-27 16:36:06,893 INFO OK: Clarkson (25 rows)
2025-08-27 16:36:16,700 INFO OK: Colgate (26 rows)
2025-08-27 16:36:24,001 INFO OK: Colorado College (27 rows)
2025-08-27 16:36

Collected 1766 rows from 3 teams.


,Current Team,Last_Name,First_Name,No,Position,Pos_OVR,Pos_Prime,Pos_Second,Yr,Ht,...,Draft_Year,NHL_Team,D_Round,Last Team,League,City,State_Province,Country,Captaincy,Shoots
0,Air Force,Clafton,Carter,35,G,G,G,None,None,"6'2""",...,None,None,None,None,None,Grand Rapids,MN,USA,None,L
1,Air Force,Hopp,Toby,37,G,G,G,None,None,"6'1""",...,None,None,None,None,None,Maple Grove,MN,USA,None,L
2,Air Force,Krick,Dylan,39,G,G,G,None,None,"6'2""",...,None,None,None,None,None,West Chester,PA,USA,None,L
3,Air Force,Spaniol,Zane,33,G,G,G,None,None,"6'3""",...,None,None,None,None,None,Ham Lake,MN,USA,None,L
4,Air Force,Wasik,Dominik,30,G,G,G,None,None,"6'1""",...,None,None,None,None,None,Superior,WI,USA,None,L
5,Air Force,Beard,Calvin,19,D,D,D,None,None,"6'2""",...,None,None,None,None,None,Southborough,MA,USA,None,R
6,Air Force,Cunningham,Nolan,20,D,D,D,None,None,"6'2""",...,None,None,None,None,None,Great Falls,MT,USA,None,R
7,Air Force,Farrell,Nate,58,D,D,D,None,None,"6'0""",...,None,None,None,None,None,Wheaton,IL,USA,None,L
8,Air Force,Hedden,Chris,22,D,D,D,None,None,"6'0""",...,None,None,None,None,None,Kalamazoo,MI,USA,None,L
9,Air Force,Houge,Simon,6,D,D,D,None,None,"5'7""",...,None,None,None,None,None,Woodbury,MN,USA,None,L


In [8]:
## Save The master CSV 

output_path = '../data/player_info/EP_master_roster_v0.1.csv'
master_df.to_csv(output_path, index=False)

In [ ]:
# #!/usr/bin/env python3
# """
# Scrape incoming freshmen major-junior stats from HockeyDB.

# Tailored to roster columns:
# ['Current Team','Last_Name','First_Name','No','Position','Yr','Ht','Wt','DOB',
#  'Hometown','Height_Inches','Draft_Year','NHL_Team','D_Round','Last Team',
#  'League','City','State_Province','Country']

# Install:
#   pip install requests beautifulsoup4 lxml pandas rapidfuzz tqdm python-dateutil

# Example:
#   python hdb_freshmen_majorjunior.py \
#     --roster roster.csv \
#     --out freshmen_majorjunior.csv \
#     --freshman-values Fr FR Freshman \
#     --limit 5 \
#     --leagues OHL WHL QMJHL

# Notes:
# - Review hockeydb.com terms/robots.txt and comply.
# - Adds polite delays + caching to be a good netizen.
# """

# from __future__ import annotations
# import argparse
# import json
# import random
# import re
# import time
# from dataclasses import dataclass
# from pathlib import Path
# from typing import Optional, List, Dict

# import pandas as pd
# import requests
# from bs4 import BeautifulSoup
# from rapidfuzz import fuzz, process
# from tqdm import tqdm
# from dateutil import parser as dtparse

# DUCKDUCKGO_HTML = "https://duckduckgo.com/html/"
# HEADERS = {
#     "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
#                   "AppleWebKit/537.36 (KHTML, like Gecko) "
#                   "Chrome/125.0.0.0 Safari/537.36",
# }

# CACHE_DIR = Path(".hdb_cache")
# CACHE_DIR.mkdir(exist_ok=True)

# def _sleep_jitter(min_s=1.2, max_s=2.8):
#     time.sleep(random.uniform(min_s, max_s))

# def _cache_path(name: str) -> Path:
#     safe = re.sub(r"[^a-zA-Z0-9_.-]+", "_", name)
#     return CACHE_DIR / f"{safe}.json"

# def _load_cache(name: str) -> Optional[dict]:
#     p = _cache_path(name)
#     if p.exists():
#         try:
#             return json.loads(p.read_text(encoding="utf-8"))
#         except Exception:
#             return None
#     return None

# def _save_cache(name: str, data: dict) -> None:
#     p = _cache_path(name)
#     p.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")

# def ddg_search_site(name: str) -> List[str]:
#     """Search for HockeyDB player pages via DuckDuckGo HTML."""
#     params = {"q": f"site:hockeydb.com {name}"}
#     resp = requests.get(DUCKDUCKGO_HTML, params=params, headers=HEADERS, timeout=25)
#     resp.raise_for_status()
#     soup = BeautifulSoup(resp.text, "lxml")
#     links = []
#     for a in soup.select("a.result__a"):
#         href = a.get("href", "")
#         if "hockeydb.com" in href and ("pdisplay" in href or "player.php?pid=" in href):
#             links.append(href)
#     # de-dup while preserving order
#     seen, uniq = set(), []
#     for u in links:
#         if u not in seen:
#             uniq.append(u); seen.add(u)
#     return uniq

# def fetch(url: str) -> str:
#     _sleep_jitter(1.0, 2.2)
#     r = requests.get(url, headers=HEADERS, timeout=30)
#     r.raise_for_status()
#     return r.text

# def parse_player_bio(soup: BeautifulSoup) -> Dict[str, str]:
#     """Grab Birth Year and Position from page text (tolerant)."""
#     text = " ".join(s.strip() for s in soup.get_text(" ").split())
#     out = {}
#     mpos = re.search(r"\b(Position|Pos)\s*[:\-]\s*([A-Za-z/]+)\b", text, re.I)
#     if mpos:
#         out["position"] = mpos.group(2).upper()
#     myear = re.search(r"\bBorn\b.*?\b(\d{4})\b", text, re.I)
#     if myear:
#         out["birth_year"] = myear.group(1)
#     return out

# def parse_stats_table(soup: BeautifulSoup) -> pd.DataFrame:
#     """Find the main stats table (Season, Team, League, GP/G/A/Pts...)."""
#     candidates = []
#     for tbl in soup.find_all("table"):
#         heads = [c.get_text(strip=True) for c in tbl.find_all("th")]
#         head_line = " | ".join(h.upper() for h in heads)
#         score = sum(int(k in head_line) for k in
#                     ["SEASON","YEAR","TEAM","LEAGUE","GP","G","A","PTS","PIM"])
#         candidates.append((score, tbl))
#     if not candidates:
#         return pd.DataFrame()

#     _, best_tbl = max(candidates, key=lambda x: x[0])
#     rows = []
#     col_names = [c.get_text(strip=True) for c in best_tbl.find_all("th")]
#     for tr in best_tbl.find_all("tr"):
#         tds = tr.find_all("td")
#         if len(tds) < 3:
#             continue
#         vals = [td.get_text(strip=True) for td in tds]
#         rows.append(vals)
#     df = pd.DataFrame(rows)
#     if col_names and len(col_names) == df.shape[1]:
#         df.columns = col_names
#     else:
#         # best-effort
#         base = ["Season","Team","League","GP","G","A","Pts","PIM"]
#         df = df.rename(columns={i: base[i] for i in range(min(len(base), df.shape[1]))})
#     return df

# @dataclass
# class PlayerHint:
#     birth_year: Optional[int] = None
#     last_team: Optional[str] = None
#     position: Optional[str] = None

# def pick_best_link(name: str, links: List[str], hint: PlayerHint) -> Optional[str]:
#     """Rank candidates by fuzzy name + optional hint checks (birth year / pos / last team)."""
#     if not links:
#         return None
#     scored = []
#     for url in links:
#         slug = url.split("/")[-1]
#         slug = re.sub(r"[-_]", " ", slug)
#         s = fuzz.token_set_ratio(name, slug)
#         if "pdisplay" in url or "player.php?pid=" in url:
#             s += 5
#         scored.append((s, url))
#     scored.sort(reverse=True)
#     top = [u for _, u in scored[:5]]

#     best, best_score = None, -1
#     for url in top:
#         try:
#             html = fetch(url)
#         except Exception:
#             continue
#         soup = BeautifulSoup(html, "lxml")
#         bio = parse_player_bio(soup)
#         stat_df = parse_stats_table(soup)

#         score = 0
#         score += fuzz.token_set_ratio(name, soup.title.get_text() if soup.title else "")
#         if hint.birth_year and bio.get("birth_year"):
#             score += 12 if str(hint.birth_year) == str(bio["birth_year"]) else 0
#         if hint.position and bio.get("position"):
#             if hint.position[0:1].upper() == bio["position"][0:1].upper():
#                 score += 6
#         if hint.last_team and not stat_df.empty and "Team" in stat_df.columns:
#             match = process.extractOne(
#                 str(hint.last_team),
#                 stat_df["Team"].astype(str).tolist(),
#                 scorer=fuzz.token_set_ratio
#             )
#             if match and match[1] >= 85:
#                 score += 8
#         if score > best_score:
#             best_score, best = score, url
#     return best or scored[0][1]

# def normalize_league(s: str) -> str:
#     return (s or "").upper().replace(".", "").strip()

# def filter_major_junior(df: pd.DataFrame, leagues_whitelist: List[str]) -> pd.DataFrame:
#     """Keep only rows whose League is in the whitelist (e.g., OHL/WHL/QMJHL)."""
#     if df.empty:
#         return df
#     colmap = {c.lower(): c for c in df.columns}
#     lg = colmap.get("league") or colmap.get("lg")
#     if not lg:
#         return pd.DataFrame()
#     tmp = df.copy()
#     tmp[lg] = tmp[lg].astype(str).map(normalize_league)
#     wl = {normalize_league(x) for x in leagues_whitelist}
#     keep = tmp[tmp[lg].isin(wl)].copy()
#     # remove playoffs/totals rows by sniffing Season col if present
#     sc = colmap.get("season") or colmap.get("year")
#     if sc:
#         keep = keep[~keep[sc].str.contains("Playoff|Total|Totals|Regular", case=False, na=False)]
#     return keep

# def scrape_player(name: str, hint: PlayerHint, leagues_whitelist: List[str]) -> Dict[str, any]:
#     """
#     Returns:
#       { name, url, birth_year, position, table:[...], error? }
#       where table is already filtered to leagues_whitelist.
#     """
#     cache_key = f"mj::{name}::{','.join(sorted(leagues_whitelist))}"
#     cached = _load_cache(cache_key)
#     if cached:
#         return cached

#     links = ddg_search_site(name)
#     if not links:
#         out = {"name": name, "url": None, "table": [], "error": "no_link_found"}
#         _save_cache(cache_key, out)
#         return out

#     url = pick_best_link(name, links, hint)
#     if not url:
#         out = {"name": name, "url": None, "table": [], "error": "no_suitable_link"}
#         _save_cache(cache_key, out)
#         return out

#     html = fetch(url)
#     soup = BeautifulSoup(html, "lxml")
#     bio = parse_player_bio(soup)
#     df = parse_stats_table(soup)

#     out = {"name": name, "url": url, "birth_year": bio.get("birth_year"), "position": bio.get("position")}
#     if df.empty:
#         out["table"] = []
#         out["error"] = "no_stats_table"
#         _save_cache(cache_key, out)
#         return out

#     # numeric cleanup best-effort
#     for col in ["GP","G","A","Pts","PIM","+/-","PPG","SHG","GWG"]:
#         if col in df.columns:
#             df[col] = pd.to_numeric(df[col].replace("", 0), errors="coerce")

#     mj = filter_major_junior(df, leagues_whitelist)
#     out["table"] = mj.to_dict(orient="records")
#     if not out["table"]:
#         out["note"] = "no_rows_in_whitelist"
#     _save_cache(cache_key, out)
#     return out

# def parse_birth_year(dob_val: str) -> Optional[int]:
#     if not dob_val or pd.isna(dob_val):
#         return None
#     try:
#         # Accept YYYY-MM-DD, MM/DD/YYYY, etc.
#         return dtparse.parse(str(dob_val), fuzzy=True).year
#     except Exception:
#         # If it's just YYYY, that’s fine
#         m = re.fullmatch(r"\s*(\d{4})\s*", str(dob_val))
#         return int(m.group(1)) if m else None

# def main():
#     ap = argparse.ArgumentParser()
#     ap.add_argument("--roster", required=True, help="Path to roster CSV")
#     ap.add_argument("--out", default="freshmen_majorjunior.csv", help="Output CSV")
#     ap.add_argument("--freshman-col", default="Yr", help="Column indicating class year")
#     ap.add_argument("--freshman-values", nargs="+", default=["Fr","FR","Freshman"],
#                     help="Values indicating freshman")
#     ap.add_argument("--first-name-col", default="First_Name")
#     ap.add_argument("--last-name-col", default="Last_Name")
#     ap.add_argument("--position-col", default="Position")
#     ap.add_argument("--dob-col", default="DOB")
#     ap.add_argument("--last-team-col", default="Last Team")
#     ap.add_argument("--leagues", nargs="+", default=["OHL","WHL","QMJHL"],
#                     help="Whitelist of major junior leagues to keep")
#     ap.add_argument("--limit", type=int, default=None,
#                     help="Max number of players to scrape (testing)")
#     args = ap.parse_args()

#     df = pd.read_csv(args.roster)
#     # Filter to freshmen
#     mask = df[args.freshman_col].astype(str).str.strip().isin(args.freshman_values)
#     freshmen = df[mask].copy()

#     if freshmen.empty:
#         print("No freshmen found with given filters.")
#         return

#     if args.limit is not None:
#         freshmen = freshmen.iloc[:args.limit].copy()

#     results = []
#     for _, row in tqdm(freshmen.iterrows(), total=len(freshmen), desc="Scraping freshmen"):
#         first = str(row.get(args.first_name_col, "")).strip()
#         last = str(row.get(args.last_name_col, "")).strip()
#         if not first or not last:
#             results.append({"Name": f"{first} {last}".strip(), "Error": "missing_name"})
#             continue
#         name = f"{first} {last}"

#         hint = PlayerHint(
#             birth_year=parse_birth_year(row.get(args.dob_col)),
#             last_team=str(row.get(args.last_team_col, "")).strip() or None,
#             position=str(row.get(args.position_col, "")).strip() or None,
#         )

#         try:
#             data = scrape_player(name, hint, leagues_whitelist=args.leagues)
#         except Exception as e:
#             data = {"name": name, "url": None, "table": [], "error": f"exception: {e}"}

#         table = data.get("table") or []
#         if not table:
#             results.append({
#                 "Name": name,
#                 "HDB_URL": data.get("url"),
#                 "BirthYear": data.get("birth_year"),
#                 "Position": data.get("position"),
#                 "Error": data.get("error", data.get("note","no_rows"))
#             })
#         else:
#             for rec in table:
#                 flat = {
#                     "Name": name,
#                     "HDB_URL": data.get("url"),
#                     "BirthYear": data.get("birth_year"),
#                     "Position": data.get("position"),
#                     # pass through roster context if helpful
#                     "Roster_Last_Team": row.get(args.last_team_col),
#                     "Roster_Current_Team": row.get("Current Team"),
#                     "Roster_Yr": row.get(args.freshman_col),
#                 }
#                 flat.update({f"stat_{k}": v for k, v in rec.items()})
#                 results.append(flat)

#     out = pd.DataFrame(results)
#     out.to_csv(args.out, index=False)
#     print(f"Saved: {args.out}")

# if __name__ == "__main__":
#     main()


In [ ]:
sample_url = 'https://www.eliteprospects.com/team/12595/brampton-steelheads/2024-2025?tab=stats' # 2034-35 Stats

sample_url = 'https://www.eliteprospects.com/team/1157/michigan-state-univ.' # Current Roster Test

### Read with pandas
import pandas as pd

dfs = pd.read_html(sample_url)
dfs[0].head(10)


In [ ]:
## Save as a temp csv file for data cleaning look

output_local = "../TEMP/elite_prospects_current_team_roster_temp.csv"
dfs[0].to_csv(output_local, index=False)
